# Working with Nested Data Structures: Solutions & Explanations

This notebook walks through the solutions to the social media post extraction exercises. The goal is not just to show *what* works, but to explain *why* and highlight common pitfalls.

---

## The Data

This structure mimics what you'll encounter when working with real social media APIs (Twitter/X, TikTok, etc.). Notice the multiple levels of nesting: dictionaries containing dictionaries, lists of dictionaries, and occasional `None` values.

In [ ]:
post_data = {
    "id": "post_12345",
    "created_at": "2024-01-15T14:30:00Z",
    "user": {
        "username": "data_journalist",
        "verified": True,
        "follower_count": 15420,
        "account_created": "2019-03-12"
    },
    "content": {
        "text": "New study on misinformation spread! Link in thread 👇",
        "language": "en",
        "hashtags": ["misinformation", "research", "datasci"],
        "mentions": ["@university_lab", "@researcher_jane"]
    },
    "engagement": {
        "likes": 342,
        "reposts": 67,
        "replies": 23,
        "views": None  # Not all platforms provide view counts
    },
    "media": [
        {
            "type": "image",
            "url": "https://example.com/img1.jpg",
            "alt_text": "Chart showing misinformation trends"
        }
    ],
    "context_annotations": {
        "topics": [
            {"name": "Media Studies", "confidence": 0.89},
            {"name": "Academic Research", "confidence": 0.76}
        ]
    }
}

---
## Q1 & Q2: Basic Nested Access

**Task:** Extract the username and post text.

**Key concept:** Chain bracket notation to drill down through layers.

In [ ]:
# Q1: Get the username
username = post_data["user"]["username"]
print(f"Username: {username}")

# Q2: Get the post text
text = post_data["content"]["text"]
print(f"Text: {text}")

**How it works:** Think of it as opening boxes inside boxes. `post_data["user"]` returns the entire user dictionary, then `["username"]` accesses the specific value within that dictionary.

---
## Q3: Handling `None` Values

**Task:** Calculate total engagement (likes + reposts + replies).

**The trap:** The `views` field is `None`. If you try to sum everything blindly, Python will complain.

In [ ]:
# This works because we explicitly pick the fields we want:
total = post_data["engagement"]["likes"] + post_data["engagement"]["reposts"] + post_data["engagement"]["replies"]
print(f"Total engagement: {total}")

In [ ]:
# More robust approach - filters out None automatically:
engagement = post_data["engagement"]
total_robust = sum(v for v in engagement.values() if v is not None)
print(f"Total engagement (robust): {total_robust}")

**Why the second approach is better:** Real API data is messy. Fields get added, removed, or are sometimes `None`. The filtering approach handles all of this gracefully.

---
## Q4: Accessing List Elements

**Task:** Get the first hashtag.

**Key concept:** Lists use numeric indices starting at 0.

In [ ]:
# Direct approach
first_hashtag = post_data["content"]["hashtags"][0]
print(f"First hashtag: {first_hashtag}")

# Safer approach (handles empty lists)
hashtags = post_data["content"]["hashtags"]
first_hashtag_safe = hashtags[0] if hashtags else None
print(f"First hashtag (safe): {first_hashtag_safe}")

**Common mistake:** Accessing `[0]` on an empty list causes an `IndexError`. The conditional expression `hashtags[0] if hashtags else None` checks if the list is non-empty first.

---
## Q5: Finding Maximum in Nested Structures

**Task:** Find the topic with the highest confidence score.

**The challenge:** We have a *list of dictionaries*, and we want the max based on a value *inside* each dictionary.

In [ ]:
# Method 1: Using max() with a key function (Pythonic)
topics = post_data["context_annotations"]["topics"]
top_topic = max(topics, key=lambda x: x["confidence"])
print(f"Top topic: {top_topic['name']} (confidence: {top_topic['confidence']})")

In [ ]:
# Method 2: Manual loop (more explicit)
topics = post_data["context_annotations"]["topics"]
max_confidence = 0
top_topic_name = None

for topic in topics:
    if topic["confidence"] > max_confidence:
        max_confidence = topic["confidence"]
        top_topic_name = topic["name"]

print(f"Top topic (loop): {top_topic_name}")

**Understanding the lambda:** `key=lambda x: x["confidence"]` tells `max()` how to compare items. It says "for each item `x`, use its confidence value for comparison."

Both methods are valid. The loop is easier to understand; the `max()` with `key` is more concise.

---
## Q6: Checking for Existence Before Accessing

**Task:** Extract alt_text from media, but handle the case where there might be no media.

In [ ]:
# Method 1: Explicit check
if post_data["media"]:  # Empty lists are "falsy"
    alt_text = post_data["media"][0]["alt_text"]
    print(f"Alt text: {alt_text}")
else:
    print("No media found")

In [ ]:
# Method 2: Try-except (ask forgiveness, not permission)
try:
    alt_text = post_data["media"][0]["alt_text"]
    print(f"Alt text: {alt_text}")
except (IndexError, KeyError):
    print("No media or alt text found")

**Which to use?** The `if` check is clearer for simple cases. `try-except` is useful when multiple things could go wrong (empty list, missing key, etc.).

---
## Q7: Processing List Items with List Comprehension

**Task:** Extract usernames from mentions, removing the @ symbol.

In [ ]:
mentions = post_data["content"]["mentions"]
print(f"Raw mentions: {mentions}")

# Method 1: replace()
usernames_v1 = [m.replace("@", "") for m in mentions]
print(f"Using replace(): {usernames_v1}")

# Method 2: lstrip() - removes from left side only
usernames_v2 = [m.lstrip("@") for m in mentions]
print(f"Using lstrip(): {usernames_v2}")

# Method 3: slicing - assumes @ is always first character
usernames_v3 = [m[1:] for m in mentions]
print(f"Using slice: {usernames_v3}")

**Trade-offs:**
- `replace("@", "")` removes ALL @ symbols (even if there's one in the middle)
- `lstrip("@")` only removes from the start (safer)
- `[1:]` assumes the @ is always there (fragile)

---
## Q8: Putting It All Together - Engagement Rate

**Task:** Calculate `(total engagement / follower count) × 100`, rounded to 2 decimals.

In [ ]:
# Robust version with all checks
engagement = post_data["engagement"]
total_engagement = sum(v for v in engagement.values() if v is not None)
follower_count = post_data["user"]["follower_count"]

if follower_count and follower_count > 0:  # Avoid division by zero
    engagement_rate = round((total_engagement / follower_count) * 100, 2)
    print(f"Engagement rate: {engagement_rate}%")
else:
    print("Cannot calculate (no followers)")

**Why all the checks?**
1. Filter out `None` from engagement values
2. Check that follower_count exists and isn't zero
3. Round to avoid ugly floating point numbers

In real research, you'll be processing thousands of posts. One bad value shouldn't crash your entire pipeline.

---
## Bonus: A Reusable Helper Function

When working with deeply nested data repeatedly, a helper function can save time and reduce errors.

In [ ]:
def get_nested_value(data, path, default=None):
    """
    Extract a value from nested dictionaries using a dot-separated path.
    
    Args:
        data: The dictionary to search
        path: Dot-separated string like "user.follower_count"
        default: Value to return if path not found
    
    Returns:
        The value at the path, or default if not found
    """
    keys = path.split(".")
    value = data
    
    try:
        for key in keys:
            value = value[key]
        return value
    except (KeyError, TypeError):
        return default

# Test it out
print(get_nested_value(post_data, "user.follower_count"))  # 15420
print(get_nested_value(post_data, "content.hashtags"))     # ['misinformation', 'research', 'datasci']
print(get_nested_value(post_data, "user.email", "N/A"))   # N/A (doesn't exist)

---
## Key Takeaways

| Lesson | Why It Matters |
|--------|----------------|
| **Always expect `None` and missing keys** | Real API data is messy |
| **Check before accessing list elements** | Empty lists cause `IndexError` |
| **Use `key=` parameter with `max()`/`min()`** | Essential for nested comparisons |
| **List comprehensions are your friend** | Concise transformations |
| **There's often multiple valid solutions** | Choose based on readability and robustness |

---
## Common Mistakes Summary

1. **Q3:** Not filtering `None` before summing
2. **Q4:** Accessing `[0]` on potentially empty lists
3. **Q5:** Trying `max(topics["confidence"])` instead of using `key=`
4. **Q6:** Not checking if media list exists before accessing
5. **Q7:** Using `[1:]` slice which fails if string doesn't start with `@`
6. **Q8:** Not checking for division by zero